# ARGUS Final — Evaluation

Load `argus_final.pt` and evaluate against UAVDT val (641 images), VisDrone held-out, KITTI held-out, and optionally GRAM-RTM.

**Before running:** Attach `blank0013/argus-final-pt` dataset (or upload `argus_final.pt` to `/kaggle/working/`). Keep `foryolotrain1/uavdt-2024-det` and `mohammadsadeqasghari/bdd100k-5class-yolov5` attached — dataset cells build the val splits and are skipped instantly on re-run via flags.

### Eval sets
| Set | Images | Domain | Source |
|---|---|---|---|
| UAVDT val | 641 | UAV overhead | merged val split |
| VisDrone held-out | ~548 | UAV elevated | locked before training |
| KITTI held-out | ~500 | Dashcam | locked before training |
| GRAM-RTM | ~5k | Spanish highway CCTV | wget (optional) |

### Metrics
Primary: **mAP50-95** · Secondary: mAP50 per class

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
from pathlib import Path
import torch, os, json as _json

NC      = 5
CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle']

WORK     = Path('/kaggle/working')
OUT_DIR  = WORK / 'argus_data'
MERGED   = OUT_DIR / 'merged'
HELD_DIR = OUT_DIR / 'held'
RUNS_DIR = WORK / 'runs'
INPUT    = Path('/kaggle/input')

for d in [MERGED/'train/images', MERGED/'train/labels',
          MERGED/'valid/images', MERGED/'valid/labels',
          HELD_DIR/'visdrone/images', HELD_DIR/'visdrone/labels',
          HELD_DIR/'kitti/images',    HELD_DIR/'kitti/labels',
          HELD_DIR/'gramrtm/images',  HELD_DIR/'gramrtm/labels',
          RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = ','.join(str(i) for i in range(torch.cuda.device_count())) or 'cpu'
BATCH  = 16
IMGSZ  = 640

# Dataset caps (used by dataset cells below)
CAP_VISDRONE = 6_471
CAP_UAVDT    = 15_000
CAP_REPLAY   = 8_000

yaml_path       = MERGED / 'data.yaml'
vis_held_yaml   = HELD_DIR / 'visdrone' / 'data.yaml'
kitti_held_yaml = HELD_DIR / 'kitti'    / 'data.yaml'
gram_held_yaml  = HELD_DIR / 'gramrtm'  / 'data.yaml'

KGL_USER = "blank0013"
KGL_KEY  = "21f03dca3d28e0e4160103e8b7735b21"
kdir = Path.home() / '.kaggle'; kdir.mkdir(exist_ok=True)
(kdir/'kaggle.json').write_text(_json.dumps({'username': KGL_USER, 'key': KGL_KEY}))
(kdir/'kaggle.json').chmod(0o600)
print(f'Kaggle creds: {KGL_USER}')
print(f'DEVICE={DEVICE}  BATCH={BATCH}  IMGSZ={IMGSZ}')

In [ ]:
# ── Install + GPU check ───────────────────────────────────────────────────────
import subprocess, sys
pkgs = ['ultralytics>=8.4.0', 'pyyaml', 'pycocotools', 'kaggle', 'pillow', 'gdown', 'tqdm']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
import torch
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {p.name}  {p.total_memory//1024**3} GB')
else:
    print('No CUDA GPU')
print(f'PyTorch {torch.__version__}')

In [ ]:
# ── Pre-flight: what's in /kaggle/input/ ──────────────────────────────────────
from pathlib import Path
import os

INPUT = Path('/kaggle/input')
print('=== /kaggle/input/ top-level ===')
for p in sorted(INPUT.iterdir()):
    try:
        n = 0
        for i, f in enumerate(p.rglob('*')):
            if f.is_file(): n += 1
            if i > 50_000: n = -1; break  # too large to count
        sz = f'{n:,} files' if n >= 0 else '>50k files'
        print(f'  {p.name:40s}  {sz}')
    except PermissionError:
        print(f'  {p.name:40s}  (no access)')

WORK = Path('/kaggle/working')
print(f'\n/kaggle/working/ contents:')
for p in sorted(WORK.iterdir()):
    if p.is_file():
        print(f'  {p.name:40s}  {p.stat().st_size//1024:8,} KB')

In [ ]:
# ── VisDrone2019-DET → YOLO ────────────────────────────────────────────────────
# ⚠ FIRST dataset cell — locks val (548 images) as held-out before training.
# Annotation format: bbox_left,bbox_top,bbox_width,bbox_height,score,category,trunc,occ
# score=0 → ignored region (skip). Category map:
#   3=bicycle→4, 4=car→0, 5=van→3, 6=truck→3, 7=tricycle→1, 9=bus→2, 10=motor→1
import os, subprocess, random
from pathlib import Path
from PIL import Image
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

VIS_MAP = {3: 4, 4: 0, 5: 3, 6: 3, 7: 1, 9: 2, 10: 1}

FLAG = OUT_DIR / '.visdrone_done'
if FLAG.exists():
    print('VisDrone: already done — skipping')
else:
    VIS_ROOT = None
    for slug in ['visdrone2019-det','visdrone-dataset','visdrone2019',
                 'shisuiotsutsuki-visdrone2019-det']:
        for base in [INPUT, INPUT/'datasets']:
            p = base / slug
            if p.exists() and any(p.rglob('*.jpg')):
                VIS_ROOT = p; print(f'VisDrone: found at {p}'); break
        if VIS_ROOT: break
    if VIS_ROOT is None:
        ds_root = INPUT / 'datasets'
        if ds_root.exists():
            for user_dir in sorted(ds_root.iterdir()):
                if not user_dir.is_dir() or VIS_ROOT: continue
                for ds_dir in sorted(user_dir.iterdir()):
                    if not ds_dir.is_dir(): continue
                    # Skip YOLO-format datasets — they have data.yaml with nc/names.
                    # UAVDT and BDD are YOLO format; VisDrone is CSV-annotated.
                    if any((ds_dir/f).exists() for f in ['data.yaml', 'dataset.yaml']):
                        continue
                    samples = list(ds_dir.rglob('annotations/*.txt'))[:3] or list(ds_dir.rglob('*.txt'))[:3]
                    if any(s.read_text(errors='ignore').split('\n')[0].count(',') >= 7 for s in samples if s.exists()):
                        VIS_ROOT = ds_dir
                        print(f'VisDrone: found (CSV format) at datasets/{user_dir.name}/{ds_dir.name}/'); break
    if VIS_ROOT is None and KGL_USER:
        RAW = WORK / '_vis_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW / '.downloaded'
        if dl_flag.exists():
            VIS_ROOT = RAW; print('VisDrone: already downloaded')
        else:
            for slug in ['shisuiotsutsuki/visdrone2019-det','aninda07/visdrone2019-det',
                         'limzl/visdrone2019-det','trainingdataset/visdrone2019-det']:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch(); VIS_ROOT = RAW
                    print(f'  Downloaded via {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:80]}')

    if VIS_ROOT is None:
        print('VisDrone: not found — CCTV eval held-out will be unavailable')
        FLAG.touch()
    else:
        def _convert_vis(ann_path, img_path):
            try:
                with Image.open(img_path) as im: W, H = im.size
            except Exception:
                W, H = 1920, 1080
            out = []
            for row in ann_path.read_text(errors='ignore').strip().split('\n'):
                if not row.strip(): continue
                pts = row.split(',')
                if len(pts) < 6: continue
                try: x1,y1,w,h,sc,cat = int(pts[0]),int(pts[1]),int(pts[2]),int(pts[3]),int(pts[4]),int(pts[5])
                except ValueError: continue
                if sc == 0 or cat not in VIS_MAP or w<=0 or h<=0: continue
                cx = max(.001,min(.999,(x1+w/2)/W)); cy = max(.001,min(.999,(y1+h/2)/H))
                bw = max(.001,min(.999,w/W));         bh = max(.001,min(.999,h/H))
                out.append(f'{VIS_MAP[cat]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
            return out

        def _find_split_dirs(root, split):
            img_d = ann_d = None
            for d in root.rglob('*'):
                if not d.is_dir(): continue
                s = split in d.name.lower() or split in str(d).lower()
                if s and (d.name=='images' or list(d.glob('*.jpg'))):
                    if img_d is None: img_d = d
                if s and (d.name=='annotations' or list(d.glob('*.txt'))):
                    if ann_d is None: ann_d = d
            return img_d, ann_d

        for split_name in ['train','val']:
            img_d, ann_d = _find_split_dirs(VIS_ROOT, split_name)
            if img_d is None: print(f'  VisDrone {split_name}: images not found'); continue
            if ann_d is None: ann_d = img_d.parent.parent / 'annotations'
            print(f'  VisDrone {split_name}: {img_d}')
            all_imgs = sorted(img_d.glob('*.*'))
            if split_name == 'val':
                hi = HELD_DIR/'visdrone'/'images'; hl = HELD_DIR/'visdrone'/'labels'
                n = 0
                for img in tqdm(all_imgs, desc='VisDrone val → held-out'):
                    ann = ann_d / (img.stem+'.txt')
                    if not ann.exists(): continue
                    lns = _convert_vis(ann, img)
                    stem = f'vis_val_{img.stem}'
                    link = hi/(stem+img.suffix)
                    if not link.exists(): _imglink(img, link)
                    (hl/(stem+'.txt')).write_text('\n'.join(lns)); n += 1
                # train: required by ultralytics validator even for val-only runs
                vis_held_yaml.write_text(
                    f'path: {(HELD_DIR/"visdrone").resolve()}\ntrain: images\nval: images\nnc: {NC}\nnames: {CLASSES}\n')
                print(f'  ✓ VisDrone val locked: {n} images')
            else:
                random.seed(42); random.shuffle(all_imgs)
                all_imgs = all_imgs[:CAP_VISDRONE]
                di = MERGED/'train'/'images'; dl = MERGED/'train'/'labels'
                n = 0
                for img in tqdm(all_imgs, desc='VisDrone train → merged'):
                    ann = ann_d / (img.stem+'.txt')
                    if not ann.exists(): continue
                    lns = _convert_vis(ann, img)
                    if not lns: continue
                    stem = f'vis_{img.stem}'
                    link = di/(stem+img.suffix)
                    if not link.exists(): _imglink(img, link)
                    (dl/(stem+'.txt')).write_text('\n'.join(lns)); n += 1
                print(f'  VisDrone train: {n:,} images → merged')
        FLAG.touch(); print('VisDrone: done')

In [ ]:
# ── UA-DETRAC: no per-frame annotations available on Kaggle — skip ────────────
# bratjay/ua-detrac-orig has images only; training XMLs are behind a registration
# wall on detrac-db.rit.albany.edu and not mirrored on Kaggle. UAVDT covers the
# overhead fixed-camera angle instead.
FLAG = OUT_DIR / '.uadetrac_done'
if not FLAG.exists():
    print('UA-DETRAC: no annotation source found — skipping')
    print('  Overhead CCTV covered by UAVDT-2024-DET instead')
    FLAG.touch()
else:
    print('UA-DETRAC: already done — skipping')

In [ ]:
# ── BDD100K 5-class → ARGUS dashcam replay ────────────────────────────────────
# Dataset: mohammadsadeqasghari/bdd100k-5class-yolov5
# Classes: 0=bus→2  1=car→0  2=motor→1  3=person→skip  4=truck→3
# Direct YOLO labels — just remap IDs, no conversion.
import os, random, yaml
from pathlib import Path
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

BDD5_MAP = {0: 2, 1: 0, 2: 1, 4: 3}   # bus car motor truck → ARGUS (person skipped)

FLAG = OUT_DIR / '.bdd_done'
if FLAG.exists():
    print('BDD100K: already done — skipping')
else:
    BDD5_ROOT = None

    # Search /kaggle/input/ and /kaggle/input/datasets/<user>/<ds>/
    for search_root in [INPUT, INPUT / 'datasets']:
        if not search_root.exists(): continue
        for p in search_root.rglob('data.yaml'):
            try:
                cfg = yaml.safe_load(p.read_text(errors='ignore'))
                names = cfg.get('names', [])
                if isinstance(names, list) and 'bus' in names and 'motor' in names and len(names) == 5:
                    root = p.parent
                    if (root/'train'/'images').exists() and (root/'train'/'labels').exists():
                        BDD5_ROOT = root
                        print(f'BDD100K 5-class: found at {root}')
                        break
            except Exception: pass
        if BDD5_ROOT: break

    if BDD5_ROOT is None and KGL_USER:
        RAW = WORK / '_bdd_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW / '.downloaded'
        if dl_flag.exists():
            BDD5_ROOT = RAW; print('BDD100K: already downloaded')
        else:
            import subprocess
            for slug in ['mohammadsadeqasghari/bdd100k-5class-yolov5',
                         'a7madmostafa/bdd100k-yolo']:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch()
                    # find root with YOLO structure
                    for p in RAW.rglob('data.yaml'):
                        try:
                            cfg = yaml.safe_load(p.read_text(errors='ignore'))
                            names = cfg.get('names',[])
                            if isinstance(names, list) and 'bus' in names and len(names)==5:
                                root = p.parent
                                if (root/'train'/'images').exists():
                                    BDD5_ROOT = root; break
                        except Exception: pass
                    if BDD5_ROOT: print(f'  Downloaded {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:80]}')

    if BDD5_ROOT is None:
        print('BDD100K: not found — skipping (attach mohammadsadeqasghari/bdd100k-5class-yolov5)')
        FLAG.touch()
    else:
        di = MERGED/'train'/'images'; dl_lbl = MERGED/'train'/'labels'
        all_collected = []
        for split in ['train', 'valid', 'test']:
            idir = BDD5_ROOT / split / 'images'
            ldir = BDD5_ROOT / split / 'labels'
            if not idir.exists(): continue
            for img in idir.glob('*.*'):
                lp = ldir / (img.stem + '.txt')
                if lp.exists(): all_collected.append((img, lp))
        random.seed(2); random.shuffle(all_collected)
        replay_pool = all_collected[:CAP_REPLAY]
        print(f'BDD100K: {len(all_collected):,} total paired, using {len(replay_pool):,} for replay')

        n = 0
        for img, lp in tqdm(replay_pool, desc='BDD5 replay → merged'):
            lines, has_m = [], False
            for row in lp.read_text(errors='ignore').strip().splitlines():
                p = row.split()
                if not p: continue
                try: orig = int(p[0])
                except ValueError: continue
                if orig in BDD5_MAP:
                    ac = BDD5_MAP[orig]
                    lines.append(f'{ac} ' + ' '.join(p[1:]))
                    if ac == 1: has_m = True
            if not lines: continue
            stem = f'bdd_{img.stem}'
            link = di / (stem + img.suffix)
            if not link.exists(): _imglink(img, link)
            (dl_lbl / (stem + '.txt')).write_text('\n'.join(lines)); n += 1
            if has_m:
                for k in range(2):
                    ml = di / (f'bdd_{img.stem}_m{k}' + img.suffix)
                    if not ml.exists(): _imglink(img, ml)
                    (dl_lbl / (f'bdd_{img.stem}_m{k}.txt')).write_text('\n'.join(lines))
        print(f'BDD replay: {n:,} base images added (moto×3)')
        FLAG.touch(); print('BDD100K: done')

In [ ]:
# ── UAVDT-2024-DET → YOLO (UAV overhead, direct YOLO labels) ─────────────────
# Dataset: foryolotrain1/uavdt-2024-det
# Classes: 0=car→0  1=truck→3  2=bus→2  3=van→0
# Already YOLO format — just remap class IDs and hardlink.
import os, random
from pathlib import Path
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

UAVDT_MAP = {0: 0, 1: 3, 2: 2, 3: 0}

FLAG = OUT_DIR / '.uavdt_done'
if FLAG.exists():
    print('UAVDT: already done — skipping')
else:
    UAVDT_ROOT = None

    # Search for UAVDT-2024-DET directory
    for p in INPUT.rglob('UAVDT-2024-DET'):
        if (p/'train'/'images').exists() and (p/'train'/'labels').exists():
            UAVDT_ROOT = p; print(f'UAVDT: found at {UAVDT_ROOT}'); break

    if UAVDT_ROOT is None and KGL_USER:
        RAW = WORK / '_uavdt_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW / '.downloaded'
        if dl_flag.exists():
            for p in RAW.rglob('UAVDT-2024-DET'):
                if (p/'train'/'images').exists():
                    UAVDT_ROOT = p; print('UAVDT: already downloaded'); break
        else:
            import subprocess
            r = subprocess.run(['kaggle','datasets','download','-d',
                                'foryolotrain1/uavdt-2024-det',
                                '-p',str(RAW),'--unzip'], capture_output=True, text=True)
            if r.returncode == 0:
                dl_flag.touch()
                for p in RAW.rglob('UAVDT-2024-DET'):
                    if (p/'train'/'images').exists():
                        UAVDT_ROOT = p; print('  Downloaded uavdt-2024-det'); break
            else:
                print(f'  uavdt-2024-det download failed: {r.stderr.strip()[:80]}')

    if UAVDT_ROOT is None:
        print('UAVDT: not found — attach foryolotrain1/uavdt-2024-det'); FLAG.touch()
    else:
        di = MERGED/'train'/'images'; dl_lbl = MERGED/'train'/'labels'
        dv_i = MERGED/'valid'/'images'; dv_l = MERGED/'valid'/'labels'
        total = 0
        for split, dst_i, dst_l in [('train', di, dl_lbl), ('val', dv_i, dv_l)]:
            idir = UAVDT_ROOT / split / 'images'
            ldir = UAVDT_ROOT / split / 'labels'
            if not idir.exists(): continue
            all_imgs = sorted(idir.glob('*.jpg'))
            if split == 'train':
                random.seed(1); random.shuffle(all_imgs)
                all_imgs = all_imgs[:CAP_UAVDT]
            for img in tqdm(all_imgs, desc=f'UAVDT {split} → merged'):
                lp = ldir / (img.stem + '.txt')
                if not lp.exists(): continue
                lines = []
                for row in lp.read_text(errors='ignore').strip().splitlines():
                    p = row.split()
                    if not p: continue
                    try: orig = int(p[0])
                    except ValueError: continue
                    if orig in UAVDT_MAP:
                        lines.append(f'{UAVDT_MAP[orig]} ' + ' '.join(p[1:]))
                if not lines: continue
                stem = f'uavdt_{img.stem}'
                link = dst_i / (stem + img.suffix)
                if not link.exists(): _imglink(img, link)
                (dst_l / (stem + '.txt')).write_text('\n'.join(lines))
                total += 1
        print(f'UAVDT: {total:,} images added to merged')
        FLAG.touch(); print('UAVDT: done')

In [ ]:
# ── KITTI → YOLO (held-out only, no wget — too large for working disk) ────────
# Only uses KITTI if attached as a Kaggle dataset in /kaggle/input/.
# leducnhuan/kitti-tracking is WRONG format (tracking, not detection) — skip it.
# For KITTI object detection: attach avg-kitti/kitti or similar manually.
import os, random
from pathlib import Path
from PIL import Image
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

FLAG = OUT_DIR / '.kitti_done'
if FLAG.exists():
    print('KITTI: already done — skipping')
else:
    IMGS = LBLS = None
    KMAP = {'Car':0, 'Van':3, 'Truck':3, 'Cyclist':1, 'Bus':2}

    # Only search /kaggle/input/ — no download
    for img_dir in INPUT.rglob('image_2'):
        lbl_dir = img_dir.parent.parent / 'label_2'
        if img_dir.is_dir() and lbl_dir.is_dir() and list(img_dir.glob('*.png')):
            IMGS = img_dir; LBLS = lbl_dir
            print(f'KITTI: found at {img_dir}'); break

    if IMGS is None:
        print('KITTI: not attached — skipping (no wget to save disk space)')
        FLAG.touch()
    else:
        all_l = sorted(LBLS.glob('*.txt'))
        random.seed(42); random.shuffle(all_l)
        kitti_held_l = all_l[:500]
        train_val_l  = all_l[500:]
        idx = int(len(train_val_l) * 0.85)

        kh_i = HELD_DIR/'kitti'/'images'; kh_l = HELD_DIR/'kitti'/'labels'
        kh_i.mkdir(parents=True, exist_ok=True); kh_l.mkdir(parents=True, exist_ok=True)
        kh_n = 0
        for lp in tqdm(kitti_held_l, desc='KITTI held-out lock'):
            ip = IMGS/(lp.stem+'.png')
            if not ip.exists(): continue
            try:
                with Image.open(ip) as im: W,H = im.size
            except Exception: continue
            lines = []
            for row in lp.read_text(errors='ignore').strip().splitlines():
                p = row.split()
                if len(p) < 15: continue
                ac = KMAP.get(p[0])
                if ac is None: continue
                x1,y1,x2,y2 = map(float, p[4:8])
                cx = max(.001,min(.999,(x1+x2)/2/W)); cy = max(.001,min(.999,(y1+y2)/2/H))
                bw = max(.001,min(.999,(x2-x1)/W));   bh = max(.001,min(.999,(y2-y1)/H))
                lines.append(f'{ac} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
            if not lines: continue
            stem = f'kitti_{lp.stem}'
            link = kh_i/(stem+'.png')
            if not link.exists(): _imglink(ip, link)
            (kh_l/(stem+'.txt')).write_text('\n'.join(lines)); kh_n += 1
        # train: required by ultralytics validator even for val-only runs
        kitti_held_yaml.write_text(
            f'path: {(HELD_DIR/"kitti").resolve()}\ntrain: images\nval: images\nnc: {NC}\nnames: {CLASSES}\n')
        print(f'  ✓ KITTI held-out locked: {kh_n} images')

        for split_l in [train_val_l[:idx], train_val_l[idx:]]:
            dst_split = 'train' if split_l is train_val_l[:idx] else 'valid'
            di = MERGED/dst_split/'images'; dl_lbl = MERGED/dst_split/'labels'
            for lp in tqdm(split_l, desc=f'KITTI {dst_split}'):
                ip = IMGS/(lp.stem+'.png')
                if not ip.exists(): continue
                try:
                    with Image.open(ip) as im: W,H = im.size
                except Exception: continue
                lines = []
                for row in lp.read_text(errors='ignore').strip().splitlines():
                    p = row.split()
                    if len(p) < 15: continue
                    ac = KMAP.get(p[0])
                    if ac is None: continue
                    x1,y1,x2,y2 = map(float, p[4:8])
                    cx = max(.001,min(.999,(x1+x2)/2/W)); cy = max(.001,min(.999,(y1+y2)/2/H))
                    bw = max(.001,min(.999,(x2-x1)/W));   bh = max(.001,min(.999,(y2-y1)/H))
                    lines.append(f'{ac} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
                if not lines: continue
                stem = f'kitti_{lp.stem}'
                link = di/(stem+'.png')
                if not link.exists(): _imglink(ip, link)
                (dl_lbl/(stem+'.txt')).write_text('\n'.join(lines))
        print(f'KITTI: training images added')
        FLAG.touch(); print('KITTI: done')

In [ ]:
# ── Dataset stats + data.yaml ────────────────────────────────────────────────
from pathlib import Path
from collections import Counter

def _count(split):
    ldir = MERGED/split/'labels'
    imgs = len(list((MERGED/split/'images').glob('*.*')))
    stats = Counter()
    for lp in ldir.glob('*.txt'):
        for row in lp.read_text(errors='ignore').splitlines():
            p = row.strip().split()
            if p:
                try: stats[int(p[0])] += 1
                except ValueError: pass
    return imgs, stats

print('='*62)
for split in ['train','valid']:
    n, stats = _count(split)
    total = sum(stats.values())
    bad   = {k:v for k,v in stats.items() if k<0 or k>=NC}
    print(f'\n{split}: {n:,} images | {total:,} boxes')
    for cid, name in enumerate(CLASSES):
        bar = '█'*min(40, int(40*stats.get(cid,0)/max(total,1)))
        print(f'  {cid} {name:12s}: {stats.get(cid,0):8,}  {bar}')
    if bad: raise ValueError(f'OUT-OF-RANGE class IDs in {split}: {bad}')
    print(f'  ✓ all IDs in [0,{NC-1}]')
print('='*62)

n_train = len(list((MERGED/'train'/'images').glob('*.*')))
assert n_train > 5_000, f'Only {n_train} training images — check dataset cells'

yaml_path.write_text(
    f'path: {MERGED.resolve()}\ntrain: train/images\nval:   valid/images\n'
    f'\nnc: {NC}\nnames: {CLASSES}\n')
print(f'\ndata.yaml → {yaml_path}')
print(yaml_path.read_text())
for yp, label in [(vis_held_yaml,'VisDrone'),(kitti_held_yaml,'KITTI')]:
    if yp.exists():
        n=len(list(yp.parent.glob('images/*.*')))
        print(f'{label} held-out: {n} images  ({yp})')

In [ ]:
# ── Load argus_final.pt ───────────────────────────────────────────────────────
# Attach blank0013/argus-final-pt as a Kaggle dataset, or upload argus_final.pt
# directly to /kaggle/working/.
from ultralytics import YOLO
from pathlib import Path
import gc, torch

_candidates = [
    INPUT / 'datasets' / 'blank0013' / 'argus-final-pt' / 'argus_final.pt',
    INPUT / 'argus-final-pt' / 'argus_final.pt',
    WORK / 'argus_final.pt',
]
MODEL_PT = next((p for p in _candidates if p.exists()), None)
assert MODEL_PT is not None, (
    "argus_final.pt not found. Attach blank0013/argus-final-pt "
    "or upload argus_final.pt to /kaggle/working/."
)
print(f'Model : {MODEL_PT}')
print(f'Size  : {MODEL_PT.stat().st_size/1024**2:.1f} MB')

model = YOLO(str(MODEL_PT))
n_params = sum(p.numel() for p in model.model.parameters())
print(f'Params: {n_params:,}  task={model.task}')
del model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Ready')

In [ ]:
# ── Evaluation — UAVDT val + VisDrone + KITTI held-out ───────────────────────
from ultralytics import YOLO
import yaml as _yaml

def _patch_yaml(yp):
    """Add 'train:' key if missing — ultralytics 8.4.x requires it even for val."""
    if not yp.exists(): return
    cfg = _yaml.safe_load(yp.read_text())
    if 'train' not in cfg:
        cfg['train'] = cfg.get('val', 'images')
        yp.write_text(_yaml.dump(cfg))

for yp in [vis_held_yaml, kitti_held_yaml, gram_held_yaml]:
    _patch_yaml(yp)

assert yaml_path.exists(), 'Run dataset cells first (Cell 9 creates data.yaml)'
model = YOLO(str(MODEL_PT))
print(f'Model: {MODEL_PT.name}  {MODEL_PT.stat().st_size/1024**2:.1f} MB  device={DEVICE}')

# ── UAVDT val ─────────────────────────────────────────────────────────────────
vm = model.val(data=str(yaml_path), imgsz=IMGSZ, batch=BATCH,
               device=DEVICE, conf=0.001, iou=0.6, verbose=False)

print('\n' + '='*64)
print('  ARGUS Final — Evaluation Results')
print('='*64)
print(f'\n  UAVDT val  (641 images · UAV overhead)')
print(f'    mAP50-95 : {vm.box.map:.4f}')
print(f'    mAP50    : {vm.box.map50:.4f}')
print(f'    P={vm.box.mp:.3f}  R={vm.box.mr:.3f}')
print(f'  Per-class:')
for name, ap50, ap in zip(CLASSES, vm.box.ap50, vm.box.maps):
    print(f'    {name:12s}:  mAP50={ap50:.4f}  mAP50-95={ap:.4f}')

# ── VisDrone held-out ─────────────────────────────────────────────────────────
if vis_held_yaml.exists():
    try:
        hm = model.val(data=str(vis_held_yaml), imgsz=IMGSZ, batch=BATCH,
                       device=DEVICE, conf=0.001, iou=0.6, verbose=False)
        n = len(list((HELD_DIR/'visdrone'/'images').glob('*.*')))
        print(f'\n  VisDrone held-out  ({n} images · UAV elevated CCTV)')
        print(f'    mAP50-95 : {hm.box.map:.4f}')
        print(f'    mAP50    : {hm.box.map50:.4f}')
        print(f'  Per-class:')
        for name, ap50, ap in zip(CLASSES, hm.box.ap50, hm.box.maps):
            print(f'    {name:12s}:  mAP50={ap50:.4f}  mAP50-95={ap:.4f}')
    except Exception as e:
        print(f'\n  VisDrone held-out: skipped — {e}')

# ── KITTI held-out ────────────────────────────────────────────────────────────
if kitti_held_yaml.exists():
    try:
        km = model.val(data=str(kitti_held_yaml), imgsz=IMGSZ, batch=BATCH,
                       device=DEVICE, conf=0.001, iou=0.6, verbose=False)
        n = len(list((HELD_DIR/'kitti'/'images').glob('*.*')))
        print(f'\n  KITTI held-out  ({n} images · dashcam)')
        print(f'    mAP50-95 : {km.box.map:.4f}')
        print(f'    mAP50    : {km.box.map50:.4f}')
    except Exception as e:
        print(f'\n  KITTI held-out: skipped — {e}')
else:
    print('\n  KITTI held-out: not available (attach KITTI detection dataset to enable)')

print('='*64)

In [ ]:
# ── GRAM-RTM eval — Spanish highway CCTV (optional wget ~2 GB) ───────────────
import subprocess, zipfile, os, xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm

GRAM_FLAG = OUT_DIR / '.gramrtm_done'
GRAM_RAW  = WORK / '_gram_raw'; GRAM_RAW.mkdir(parents=True, exist_ok=True)
GRAM_URL  = 'https://gram.web.uah.es/data/datasets/rtm/GRAM-RTMv4.zip'
GRAM_ZIP  = GRAM_RAW / 'GRAM-RTMv4.zip'

if not GRAM_FLAG.exists():
    if not GRAM_ZIP.exists():
        r = subprocess.run(['wget','-q','--show-progress','-O',str(GRAM_ZIP),GRAM_URL],
                           capture_output=False, check=False)
        if r.returncode != 0:
            print(f'GRAM-RTM wget failed (rc={r.returncode}) — skipping'); GRAM_ZIP = None
    if GRAM_ZIP and GRAM_ZIP.exists():
        with zipfile.ZipFile(GRAM_ZIP) as zf: zf.extractall(GRAM_RAW)
        GRAM_ZIP.unlink(missing_ok=True); GRAM_FLAG.touch()

if not GRAM_FLAG.exists():
    print('GRAM-RTM: not available (wget failed or skipped)')
else:
    GRAM_MAP = {'car':0,'truck':3,'van':0,'big-truck':3,'bus':2,'motorcycle':1,'bicycle':4}
    hi = HELD_DIR/'gramrtm'/'images'; hl = HELD_DIR/'gramrtm'/'labels'
    hi.mkdir(parents=True, exist_ok=True); hl.mkdir(parents=True, exist_ok=True)
    xml_files = sorted(GRAM_RAW.rglob('*.xml'))
    print(f'GRAM-RTM: {len(xml_files)} XMLs found')
    n = 0
    for xp in tqdm(xml_files[:5000], desc='GRAM-RTM → YOLO'):
        try: root_el = ET.parse(xp).getroot()
        except ET.ParseError: continue
        sz = root_el.find('size')
        W = int(sz.find('width').text) if sz else 1920
        H = int(sz.find('height').text) if sz else 1080
        lines = []
        for obj in root_el.findall('object'):
            name_el = obj.find('name')
            if name_el is None: continue
            ac = GRAM_MAP.get(name_el.text.lower().strip())
            if ac is None: continue
            bb = obj.find('bndbox')
            if bb is None: continue
            try:
                x1=float(bb.find('xmin').text); y1=float(bb.find('ymin').text)
                x2=float(bb.find('xmax').text); y2=float(bb.find('ymax').text)
            except (TypeError,ValueError): continue
            cx=max(.001,min(.999,(x1+x2)/2/W)); cy=max(.001,min(.999,(y1+y2)/2/H))
            bw=max(.001,min(.999,(x2-x1)/W));   bh=max(.001,min(.999,(y2-y1)/H))
            lines.append(f'{ac} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        if not lines: continue
        img_p = xp.with_suffix('.jpg')
        if not img_p.exists(): img_p = xp.with_suffix('.png')
        if not img_p.exists(): continue
        stem = f'gram_{xp.stem}'
        link = hi/(stem+img_p.suffix)
        if not link.exists():
            try: os.link(img_p, link)
            except OSError: os.symlink(img_p.resolve(), link)
        (hl/(stem+'.txt')).write_text('\n'.join(lines)); n += 1

    if n >= 10:
        gram_held_yaml.write_text(
            f'path: {(HELD_DIR/"gramrtm").resolve()}\ntrain: images\nval: images\nnc: {NC}\nnames: {CLASSES}\n')
        try:
            gm = model.val(data=str(gram_held_yaml), imgsz=IMGSZ, batch=8,
                           device=DEVICE, conf=0.001, iou=0.6, verbose=False)
            print(f'\nGRAM-RTM ({n} images · Spanish highway CCTV)')
            print(f'  mAP50-95 : {gm.box.map:.4f}')
            print(f'  mAP50    : {gm.box.map50:.4f}')
        except Exception as e:
            print(f'GRAM-RTM eval failed: {e}')
    else:
        print(f'GRAM-RTM: only {n} images converted')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('='*64)
print('  ARGUS Final — Complete')
print('='*64)
print(f'  Model : {MODEL_PT}')
print(f'  Size  : {MODEL_PT.stat().st_size/1024**2:.1f} MB')
print()
print('  Deploy:')
print('    from ultralytics import YOLO')
print('    model = YOLO("argus_final.pt")')
print('    results = model("video.mp4")')
print('='*64)